# Сборный проект — 2. HR-аналитика
## Описание проекта


HR-аналитики компании «Работа с заботой» помогают бизнесу оптимизировать управление персоналом: бизнес предоставляет данные, а аналитики предлагают, как избежать финансовых потерь и оттока сотрудников. В этом HR-аналитикам пригодится машинное обучение, с помощью которого получится быстрее и точнее отвечать на вопросы бизнеса.
Компания предоставила данные с характеристиками сотрудников компании. Среди них — уровень удовлетворённости сотрудника работой в компании. Эту информацию получили из форм обратной связи: сотрудники заполняют тест-опросник, и по его результатам рассчитывается доля их удовлетворённости от 0 до 1, где 0 — совершенно неудовлетворён, 1 — полностью удовлетвоён.

### Задачи:

1. Построить модель, которая сможет предсказать уровень удовлетворённости сотрудника на основе данных заказчика.
2. Построить модель, которая сможет на основе данных заказчика предсказать то, что сотрудник уволится из компании.


## Цель исследования

1. Построить и сравнить модели регрессии для **прогноза уровня удовлетворённости сотрудника работой** (`job_satisfaction_rate`) по данным компании. Критерий успеха: **SMAPE ≤ 15%** на тестовой выборке.
2. Построить и сравнить модели классификации для **прогноза увольнения сотрудника** (`quit`). Критерий успеха: **ROC-AUC ≥ 0.91** на тестовой выборке.
3. Проверить гипотезу о взаимосвязи удовлетворённости и увольнения: сравнить распределения `job_satisfaction_rate` для ушедших и оставшихся сотрудников и **использовать предсказанную удовлетворённость** как дополнительный признак во второй задаче.
4. Выявить **ключевые факторы**, влияющие на удовлетворённость и риск увольнения (важности признаков, целевые группировки) и сформулировать **практические рекомендации** для HR-аналитиков: куда направить усилия (нагрузка, зарплата, обратная связь, промо), чтобы снизить риск оттока.
5. Обеспечить **воспроизводимость и отсутствие утечки данных**: вся подготовка внутри `Pipeline`/`ColumnTransformer`; при добавлении `job_satisfaction_rate` во 2-ю задачу использовать **OOF-предсказания** на train и «честные» предсказания на test.

**Исходные данные** (CSV):
- `train_job_satisfaction_rate.csv` — обучающая выборка для регрессии (цель: `job_satisfaction_rate`);
- `test_features.csv` — признаки тестовой выборки (общие для обеих задач);
- `test_target_job_satisfaction_rate.csv` — целевой признак регрессии для теста;
- `train_quit.csv` — обучающая выборка для классификации (цель: `quit`);
- `test_target_quit.csv` — целевой признак классификации для теста.


### Этапы исследования

1. **Обзор и изучение данных**  
   - Проверю состав столбцов, типы данных, диапазоны, уникальные значения, дубликаты.  
   - Сопоставлю `id` между наборами, выровняю тестовые таргеты по `id`.

2. **Исследовательский анализ данных (EDA)**  
   - Распределения численных/категориальных признаков, выбросы, группировки по отделам и уровням.  
   - Предварительные гипотезы: влияние загруженности, стажа, оценки руководителя и зарплаты на удовлетворённость; отдел/уровень — на риск увольнения.
   
3. **Предобработка**  
   - Обработка пропусков в **пайплайне** (`SimpleImputer`), без ручных трансформаций вне `Pipeline`.  
   - Кодирование категориальных признаков как минимум **двумя стратегиями** (например, `OrdinalEncoder` для рангового `level` и `OneHotEncoder` для `dept`/других категориальных).  
   - Масштабирование числовых признаков (`StandardScaler`) и аккуратное обращение с бинарными (0/1 либо как категории — последовательно во всех моделях).  
   - Разделение на фолды/валидацию через `KFold/StratifiedKFold` (фиксированный `random_state`) — без утечки.


4. **Моделирование регрессии (прогноз `job_satisfaction_rate`)**  
   - Базовые кандидаты: **линейная модель с регуляризацией** (`Ridge`) и **дерево решений** (`DecisionTreeRegressor`).  
   - Подбор гиперпараметров минимум для одной модели через `GridSearchCV`, **метрика — SMAPE**.  
   - Сравнение на валидации и тесте; выбор лучшей модели и фиксация результата (целевой порог **≤ 15%**).

5. **Проверка связи удовлетворённости и увольнения**  
   - На тесте объединю `test_features` с `test_target_job_satisfaction_rate` и `test_target_quit` по `id`.  
   - Сравню распределения `job_satisfaction_rate` у ушедших и оставшихся, сделаю краткий вывод.

6. **Моделирование классификации (прогноз `quit`)**  
   - Кандидаты: **LogisticRegression**, **RandomForestClassifier**, **DecisionTreeClassifier**.  
   - Подбор гиперпараметров **минимум для двух** моделей по **ROC-AUC** (CV).  
   - Итоговая проверка на тесте, целевой порог **ROC-AUC ≥ 0.91**.  
   - (Опционально) отбор признаков/важности признаков для интерпретации.

7. **Сравнение и оценка моделей**  
   - Для регрессии: SMAPE на CV и тесте; для классификации ROC-AUC. 
   - Сводная таблица результатов с лучшими конфигурациями гиперпараметров.

8. **Интерпретация и рекомендации для HR**  
   - Топ-факторы, влияющие на удовлетворённость и риск ухода (важности/коэффициенты/частоты по группам).  
   - Практические рекомендации.




## Импорт

In [ ]:
# Стандартные библиотеки
import logging
import time
from pathlib import Path

# Вычисления и обработка данных
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns


# Phik для корреляционного анализа
import phik
from phik import resources
from phik.report import plot_correlation_matrix


# Scikit-learn: модели, метрики, трансформеры и утилиты
import sklearn # для проверки 
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import auc, make_scorer, roc_auc_score, roc_curve
from sklearn.model_selection import (GridSearchCV, KFold, RandomizedSearchCV,
									 StratifiedKFold)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, LabelEncoder,
								   OneHotEncoder, OrdinalEncoder,
								   StandardScaler)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# Установка RANDOM_STATE
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Директория для сохранения изображений
IMG_DIR = Path("../images")
IMG_DIR.mkdir(exist_ok=True)

def save_plot(name: str, dpi: int = 300):
    plt.tight_layout()
    plt.savefig(IMG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight")
    print(f"Saved: {IMG_DIR / f'{name}.png'}")


## Задача 1: предсказание уровня удовлетворённости сотрудника


Для этой задачи заказчик предоставил данные с признаками:
* `id` — уникальный идентификатор + сотрудника;
* `dept` — отдел, в котором работает сотрудник;
* `level` — уровень занимаемой должности;
* `workload` — уровень загруженности сотрудника;
* `employment_years` — длительность работы в компании (в годах);
* `last_year_promo` — показывает, было ли повышение за последний год;
* `last_year_violations` — показывает, нарушал ли сотрудник трудовой договор за последний год;
* `supervisor_evaluation` — оценка качества работы сотрудника, которую дал руководитель;
* `salary` — ежемесячная зарплата сотрудника;
* `job_satisfaction_rate` — уровень удовлетворённости сотрудника работой в компании, целевой признак.

### Загрузка данных

In [ ]:
# --- Загрузка исходных данных ---

# Возможные пути, где могут находиться датасеты
BASE_DIRS = [Path("/datasets"), Path("datasets"), Path("../data"), Path(".")]

# Имена файлов, переданных заказчиком
FILES = {
    "train_js": "train_job_satisfaction_rate.csv",
    "test_features": "test_features.csv",
    "test_target_js": "test_target_job_satisfaction_rate.csv",
}

def load_csv_files(files_cfg, base_dirs=BASE_DIRS):
    """
    Загружает все CSV-файлы из указанных путей.
    Проверяет наличие, выводит форму датафрейма или сообщение об ошибке.
    """
    dfs = {}
    for key, filename in files_cfg.items():
        # ищем первый существующий путь
        path = next((base / filename for base in base_dirs if (base / filename).exists()), None)
        if path:
            try:
                dfs[key] = pd.read_csv(path)
                print(f"✅ {key}: {path} — shape={dfs[key].shape}")
            except Exception as e:
                print(f"⚠️ Ошибка при загрузке {path}: {e}")
                dfs[key] = None
        else:
            print(f"⚠️ {key}: {filename} не найден")
            dfs[key] = None
    return dfs


# --- Выполнение загрузки ---

dfs = load_csv_files(FILES)

# Присвоение переменным для удобства дальнейшей работы
train_js = dfs.get("train_js")
test_features = dfs.get("test_features")
test_target_js = dfs.get("test_target_js")

# Просмотр нескольких первых строк каждого датафрейма
for key, df in dfs.items():
    if df is not None:
        print(f"📄 {key}:")
        display(df.head(3))


In [ ]:
# Первичный осмотр данных:
# - .info(): количество строк, пропуски, типы данных
# - .head(): первые 5 строк таблицы

for name, df in dfs.items():
	if df is not None:
		print("="*60)
		print(f"\n{name.upper()} — INFO:")
		df.info()
		print(f"\n{name.upper()} — HEAD:")
		display(df.head())


## Выводы после первичного осмотра данных (Задача 1)

- Все необходимые файлы для первой задачи успешно загружены:
  - `train_job_satisfaction_rate.csv` — обучающая выборка (4000 записей, 10 признаков);
  - `test_features.csv` — тестовая выборка (2000 записей, 9 признаков);
  - `test_target_job_satisfaction_rate.csv` — целевой признак для теста (2000 записей).

- **Структура данных совпадает** между обучающей и тестовой выборками, отличия только в наличии целевого признака `job_satisfaction_rate` в train-файле.

- **Типы признаков:**
  - Категориальные (`object`): `dept`, `level`, `workload`, `last_year_promo`, `last_year_violations`;
  - Числовые (`int64`, `float64`): `employment_years`, `supervisor_evaluation`, `salary`, а также целевой признак `job_satisfaction_rate`.

- **Пропуски:**  
  Незначительное количество пропусков (< 0.2 %) обнаружено в признаках `dept` и `level`.  
  План: заполнить модой в пайплайне при подготовке данных.

- **Типы данных корректны**, числовые признаки не содержат пропусков, целевой признак находится в диапазоне 0–1.

- **Общее впечатление:**  
  Данные чистые, сбалансированные по структуре, хорошо подходят для задач регрессии.  
  Большинство признаков категориальные — потребуется кодирование (OneHotEncoder и/или OrdinalEncoder) перед обучением модели.

**Следующий шаг:** провести предобработку данных — проверить пропуски, типы признаков и при необходимости внести минимальные изменения.  
Заполнение пропусков будет выполнено на этапе пайплайна.


### Предобработка данных


In [ ]:
# Проверка пропущенных значений для всех таблиц
for name, df in dfs.items():
    if df is not None:
        print(f"\n{name.upper()} — MISSING VALUES:")
        print(df.isnull().sum())


In [ ]:
# Проверка на дубликаты
for name, df in dfs.items():
    if df is not None:
        print(f"\n{name.upper()} — DUPLICATES:")
        print(df.duplicated().sum())

In [ ]:
# Проверка типов данных
for name, df in dfs.items():
	if df is not None:
		print(f"\n{name.upper()} — DATA TYPES:")
		print(df.dtypes)

## Вывод по предобработке данных

### Проверка пропусков
- В признаках `dept` и `level` обнаружено небольшое количество пропусков:  
  - `dept`: 6 строк в обучающей и 2 в тестовой выборках  
  - `level`: 4 строки в обучающей и 1 в тестовой выборках  
- Остальные признаки не содержат пропусков.  
- Целевой признак `job_satisfaction_rate` заполнен полностью.

### Проверка на дубликаты
- Дубликаты отсутствуют во всех таблицах (`train_js`, `test_features`, `test_target_js`).

### Проверка типов данных
- Категориальные (`object`):  
  `dept`, `level`, `workload`, `last_year_promo`, `last_year_violations`.  
- Числовые (`int64`, `float64`):  
  `employment_years`, `supervisor_evaluation`, `salary`, `job_satisfaction_rate`.  
- Типы данных корректны, числовые признаки не содержат пропусков.  
- Целевой признак `job_satisfaction_rate` имеет диапазон значений от 0 до 1.

### Вывод
- Пропуски присутствуют только в двух категориальных признаках (`dept`, `level`) и **будут заполнены в пайплайне** (модой).  
- Дубликатов нет, типы данных корректны.  
- Набор данных чистый и готов к дальнейшему анализу.

**Следующий шаг:** провести исследовательский анализ данных (EDA) — изучить распределения признаков и их взаимосвязь с целевым показателем `job_satisfaction_rate`.


### Исследовательский анализ данных
Исследуйте все признаки и сделайте выводы о том, как их нужно подготовить.


In [ ]:
# === EDA  ===

sns.set(style="whitegrid", palette="muted", font_scale=1.2)

# --- списки признаков ---
num_features = ['employment_years', 'supervisor_evaluation', 'salary']
cat_features = ['dept', 'level', 'workload', 'last_year_promo', 'last_year_violations']
target = 'job_satisfaction_rate'

# --- подготавливаем общий df для сравнения train/test ---
df_train = train_js.copy()
df_test = test_features.copy()
df_train['dataset'] = 'train'
df_test['dataset'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

# дискретные числовые (по требованию countplot)
discrete_num = {'employment_years', 'supervisor_evaluation'}

# --- 1. Числовые распределения: дискретные — countplot, остальные — histplot ---
for col in num_features:
	plt.figure(figsize=(8, 5))
	if col in discrete_num:
		order = sorted(df_all[col].dropna().unique())
		sns.countplot(data=df_all, x=col, hue='dataset', order=order)
		plt.ylabel('Количество')
	else:
		sns.histplot(data=df_all, x=col, hue='dataset', bins=30, kde=True,
					 stat='density', common_norm=False, edgecolor='black')
		plt.ylabel('Плотность')
	plt.title(f'Распределение: {col} (train vs test)')
	plt.xlabel(col)
	plt.grid(axis='y', linestyle='--', alpha=0.7)
	plt.tight_layout()
	save_plot(f"distribution_{col}_train_vs_test")
	plt.show()

# --- 2. Категориальные распределения (общее распределение категорий) ---
for col in cat_features:
	plt.figure(figsize=(10, 5))
	order = df_all[col].value_counts().index
	sns.countplot(data=df_all, x=col, hue='dataset', order=order)
	plt.title(f'Категории (train vs test): {col}')
	plt.xlabel(col)
	plt.ylabel('Количество')
	plt.xticks(rotation=30)
	plt.tight_layout()
	save_plot(f"categories_{col}_train_vs_test")
	plt.show()

# --- 3. Boxplot: распределение таргета по категориям (на train) ---
for col in cat_features:
	plt.figure(figsize=(10, 5))
	sns.boxplot(data=df_train, x=col, y=target, palette='pastel')
	plt.title(f'Удовлетворённость по {col} (train)')
	plt.xlabel(col)
	plt.ylabel('Уровень удовлетворённости')
	plt.xticks(rotation=30)
	plt.grid(axis='y', linestyle='--', alpha=0.7)
	plt.tight_layout()
	save_plot(f"{target}_by_{col}_train")
	plt.show()

# === 4. phik: связи с таргетом и мультиколлинеарность (train) ===

# 4.0 Формируем списки для interval_cols
continuous_num = [c for c in num_features if c not in discrete_num]

# Для матрицы с таргетом: интервальные = непрерывные числовые + таргет
interval_with_target = continuous_num + [target]

# 4.1 Предикторы + таргет
cols_full = num_features + cat_features + [target]
phik_full = df_train[cols_full].phik_matrix(interval_cols=interval_with_target)

plt.figure(figsize=(14, 6))
sns.heatmap(phik_full, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt=".2f")
plt.title('Матрица phik (предикторы + таргет) — train')
plt.tight_layout()
save_plot("phik_matrix_predictors_and_target_train")
plt.show()

# 4.2 Только предикторы (мультиколлинеарность): интервальные = только непрерывные числовые
cols_pred = num_features + cat_features
phik_pred = df_train[cols_pred].phik_matrix(interval_cols=continuous_num)

plt.figure(figsize=(14, 8))
sns.heatmap(phik_pred, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt=".2f")
plt.title('Корреляции phik между предикторами — train')
plt.tight_layout()
save_plot("phik_matrix_predictors_train")
plt.show()



Для матрицы `phik` с таргетом учтено, что `employment_years` и `supervisor_evaluation` — **дискретные признаки**, поэтому они исключены из `interval_cols`.  
В `interval_cols` добавлен `job_satisfaction_rate`, как интервальный признак.  
Теперь Phik отражает **категориальные различия**, а не численные интервалы.

Наиболее сильная связь наблюдается между `supervisor_evaluation` и `job_satisfaction_rate` (0.76) — высокая оценка руководителя связана с большей удовлетворённостью.  
Связь `last_year_violations` с таргетом умеренная отрицательная (0.56): нарушения коррелируют с понижением удовлетворённости.


In [ ]:
# --- 5. Короткий текстовый снэпшот распределений (train vs test) без статистических тестов ---
print("\n=== Краткий снэпшот распределений (top-5 долей значений) ===")
for name, ds in [('train', df_train), ('test', df_test)]:
	print(f"\n[{name}]")
	for col in num_features + cat_features:
		vc = ds[col].value_counts(normalize=True).head()
		print(f"{col}:\n{vc}\n")

# --- Scatterplots: взаимосвязь числовых признаков и таргета ---
for col in num_features:
    plt.figure(figsize=(6,5))
    sns.scatterplot(data=train_js, x=col, y='job_satisfaction_rate', alpha=0.6)
    sns.regplot(data=train_js, x=col, y='job_satisfaction_rate', scatter=False, color='red', ci=None)
    plt.title(f'{col} vs job_satisfaction_rate')
    plt.xlabel(col)
    plt.ylabel('job_satisfaction_rate')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    save_plot(f"{col}_vs_job_satisfaction_rate")
    plt.show()

## Исследовательский анализ данных (EDA)

### Числовые признаки
- **`employment_years`** — распределение смещено влево: большинство сотрудников работают 1–3 года.  
  Это может говорить о высокой текучести кадров или расширении штата.  
  С увеличением стажа наблюдается лёгкий рост удовлетворённости (**phik = 0.33**).  
- **`supervisor_evaluation`** — значения сосредоточены на уровнях 3 и 4 («средне» – «выше среднего»).  
  Сильная положительная связь с таргетом (**phik = 0.76**) — один из ключевых факторов влияния на удовлетворённость.  
- **`salary`** — распределение правостороннее (основная масса — 20–40 тыс.), что характерно для реальных зарплатных данных.  
  Связь с удовлетворённостью слабая, но положительная (**phik = 0.17**).  
  Выбросов не обнаружено.

### Корреляции
- Наиболее сильные связи с целевым признаком `job_satisfaction_rate` отмечаются у:
  - `supervisor_evaluation` (**phik = 0.76**) — чем выше оценка руководителя, тем выше удовлетворённость;  
  - `last_year_violations` (**phik = 0.56**, обратная зависимость) — наличие нарушений снижает удовлетворённость;  
  - `employment_years` (**phik = 0.33**) — небольшой рост удовлетворённости с опытом работы.  
- Между предикторами заметны умеренные взаимосвязи (до **phik ≈ 0.79**), что говорит о **умеренной, но не критичной мультиколлинеарности**.  
  Наиболее тесно коррелируют `salary`, `level` и `workload`, что отражает карьерное и нагрузочное различие между уровнями сотрудников.

### Категориальные признаки
- **`dept`** — различия между отделами несущественны: у `technology` удовлетворённость немного выше, у `sales` — чуть ниже.  
- **`level`** — удовлетворённость растёт от `junior` к `senior`, что логично отражает накопление опыта и привилегий.  
- **`workload`** — при высокой нагрузке медианная удовлетворённость снижается.  
- **`last_year_promo`** — повышение в прошлом году коррелирует с заметным ростом удовлетворённости.  
- **`last_year_violations`** — наличие дисциплинарных нарушений сопровождается значительно меньшей удовлетворённостью.

### Сравнение train/test
- Распределения числовых и категориальных признаков в train и test схожи; существенных смещений не выявлено.  
  Это подтверждает корректность разбиения и сопоставимость выборок.

### Вывод
- Наиболее информативные признаки: `supervisor_evaluation`, `last_year_violations`, `last_year_promo`, `level`, `salary`.  
- Для подготовки данных потребуется:
  - кодирование категориальных признаков (`OneHotEncoder` / `OrdinalEncoder`);  
  - масштабирование числовых признаков (`StandardScaler` / `MinMaxScaler`);  
  - заполнение пропусков в `dept` и `level` с помощью моды в пайплайне.

**Следующий шаг — предобработка данных и построение моделей регрессии для прогноза `job_satisfaction_rate`.**


### Подготовка данных




Создадим пайплайн предобработки, чтобы автоматизировать все шаги подготовки данных прямо внутри модели.  
Числовые признаки масштабируются, категориальные кодируются (упорядоченные через `OrdinalEncoder`, остальные через `OneHotEncoder`).  
Это нужно, чтобы модели получали только числовой формат и не было утечки данных между обучением и тестом.


In [ ]:
# Совместимо со sklearn==0.24.1

# --- признаки ---
num_features = ['employment_years', 'supervisor_evaluation', 'salary']
ordinal_features = ['level', 'workload']  # ранговый
nominal_features = ['dept', 'last_year_promo', 'last_year_violations']  # номинальные

In [ ]:
# 1) Строковый клинер как штатный трансформер (trim + lower + '' -> NaN)
def _clean_strings(df):
    df = df.copy()
    for c in df.columns:
        s = df[c].astype(str).str.strip().str.lower()
        s = s.replace('', np.nan)  # фикс кейса ' '
        df[c] = s
    return df

string_cleaner = FunctionTransformer(_clean_strings, validate=False)

# 2) Backward-compatible OHE с drop='first' (защита от dummy trap)
def make_ohe_drop_first():
    try:
        # в sklearn>=1.2 есть sparse_output
        return OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
    except TypeError:
        # для sklearn<=1.1 (в т.ч. 0.24.1) используем sparse
        return OneHotEncoder(handle_unknown='ignore', drop='first', sparse=False)

# 3) Пайплайны
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

ordinal_transformer = Pipeline([
    ('clean',   string_cleaner),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ord',     OrdinalEncoder(
        categories=[['junior', 'middle', 'senior'], ['low', 'medium', 'high']],   
        handle_unknown='use_encoded_value',
        unknown_value=-1                               # новые/неизвестные уровни
    ))
])

nominal_transformer = Pipeline([
    ('clean',   string_cleaner),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  make_ohe_drop_first())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer,     num_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features)
])

# НЕ нужно больше clean_df(...) снаружи!
X_train = train_js.drop(columns=['job_satisfaction_rate', 'id'])
y_train = train_js['job_satisfaction_rate']

X_prepared = preprocessor.fit_transform(X_train)
print(f"Размер матрицы после трансформации: {X_prepared.shape}")

In [ ]:
# --- Проверим наличие дубликатов в train после удаления id ---
if 'id' in train_js.columns:
	train_no_id = train_js.drop(columns=['id'])
else:
	train_no_id = train_js.copy()
	print("⚠️ Колонка 'id' отсутствует, используем исходный DataFrame.")

duplicates_count = train_no_id.duplicated().sum()
print(f"Количество дубликатов после удаления id: {duplicates_count}")

if duplicates_count > 0:
	# Удаляем дубликаты и сбрасываем индексы
	train_no_id = train_no_id.drop_duplicates().reset_index(drop=True)
	print(f"После удаления дубликатов: {train_no_id.shape[0]} строк (изначально {train_js.shape[0]})")

	# Обновляем train_js только если были дубликаты
	train_js = train_no_id.copy()
	print("Дубликаты удалены, данные обновлены.")
else:
	print("Дубликатов не найдено, данные не изменены.")


## Подготовка данных

Для подготовки данных использован `ColumnTransformer`, который объединяет разные стратегии обработки признаков:

- **Числовые признаки** (`employment_years`, `supervisor_evaluation`, `salary`):
  - заполнение пропусков медианой;
  - масштабирование (`StandardScaler`).

- **Категориальные признаки**:
  - `level` — закодирован с помощью `OrdinalEncoder`, так как категории упорядочены (`junior` < `middle` < `senior`);
  - `dept`, `workload`, `last_year_promo`, `last_year_violations` — закодированы с помощью `OneHotEncoder`.

Пропуски в категориальных признаках заполнены **модой** (`SimpleImputer(strategy='most_frequent')`).

В результате создан единый трансформер `preprocessor`, который будет встроен в пайплайн моделей.

**Следующий шаг:** обучение и сравнение моделей (линейная регрессия и дерево решений) для предсказания `job_satisfaction_rate`.


### Обучение моделей

#### Мини-бейзлайн перед обучением

Перед обучением я провела короткий бейзлайн-анализ — сравнила базовые модели  
на одинаковых условиях (`KFold`, общая метрика, единый препроцессинг).  
Это помогает оценить стабильность результатов и избежать переобучения до тюнинга.

---

#### Обучение моделей

По заданию необходимо обучить **две модели** — линейную и дерево решений —  
и выбрать лучшую по метрике **SMAPE**  
(критерий успеха: SMAPE ≤ 15 на тестовой выборке).

- **Ridge Regression** — линейная модель с регуляризацией, устойчива к мультиколлинеарности после One-Hot кодирования;  
- **DecisionTreeRegressor** — учитывает нелинейные зависимости и взаимодействия признаков.  

Метрика **SMAPE (Symmetric Mean Absolute Percentage Error)** задана в условиях проекта.  
Она выражает ошибку прогноза в процентах и позволяет сравнивать качество моделей при разных масштабах данных.  

Подбор гиперпараметров выполнялся через `GridSearchCV`:  
- Ridge — `alpha` (регуляризация);  
- Decision Tree — `max_depth`, `min_samples_split`, `min_samples_leaf`, `ccp_alpha`.  

Кросс-валидация `KFold(5, shuffle=True, random_state=42)` обеспечивает надёжную оценку без утечек данных.





In [ ]:
# Настройка логирования
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- 1) Определение метрики SMAPE ---
def smape(y_true, y_pred):
	"""
	Вычисление симметричного среднего абсолютного процентного отклонения (SMAPE).
	"""
	y_true, y_pred = np.array(y_true), np.array(y_pred)
	y_pred = np.clip(y_pred, 0, 1)  # Ограничиваем предсказания в диапазоне [0, 1]
	denom = (np.abs(y_true) + np.abs(y_pred)) / 2
	denom = np.where(denom == 0, 1e-9, denom)  # Избегаем деления на ноль
	return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)

# Создаем объект для использования SMAPE в GridSearchCV
smape_scorer = make_scorer(smape, greater_is_better=False)

In [ ]:
# --- 2) Разделение данных ---
# --- 2) Разделение и выравнивание данных ---
# Признаки и таргет на train (удалим 'id' только если он есть)
X_train = train_js.drop(columns=[c for c in ['id', 'job_satisfaction_rate'] if c in train_js.columns]).copy()
y_train = train_js['job_satisfaction_rate'].copy()

# Корректно выравниваем test по id, но если id уже дропнули — просто берём по текущему порядку
if ('id' in getattr(test_features, 'columns', [])) and ('id' in getattr(test_target_js, 'columns', [])):
    test_join = test_features.merge(
        test_target_js[['id', 'job_satisfaction_rate']],
        on='id', how='left', validate='one_to_one'
    ).reset_index(drop=True)
    X_test = test_join.drop(columns=['id', 'job_satisfaction_rate'])
    y_test = test_join['job_satisfaction_rate']
else:
    # fall-back без id: считаем, что порядок уже согласован
    X_test = test_features.reset_index(drop=True).copy()
    y_test = test_target_js['job_satisfaction_rate'].reset_index(drop=True)

# sanity-check
assert len(X_test) == len(y_test), "Длины X_test и y_test не совпадают"
print(f"X_train: {X_train.shape}, y_train: {y_train.shape[0]}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape[0]}")


# --- 3) Кросс-валидация ---
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- 4) Функция для обучения и оценки модели ---
def train_and_evaluate_model(pipe, param_grid, model_name):
	"""
	Обучение и оценка модели с использованием GridSearchCV.
	"""
	search = GridSearchCV(
		pipe,
		param_grid=param_grid,
		scoring=smape_scorer,
		cv=cv,
		n_jobs=-1
	)
	search.fit(X_train, y_train)

	best_model = search.best_estimator_
	cv_score = abs(search.best_score_)
	test_pred = best_model.predict(X_test)
	test_score = smape(y_test, test_pred)

	logging.info(f"✅ {model_name} — лучшие параметры: {search.best_params_}")
	logging.info(f"{model_name} SMAPE (CV):   {cv_score:.2f}")
	logging.info(f"{model_name} SMAPE (test): {test_score:.2f}")

	return model_name, test_score

# --- 5) Модель 1: линейная регрессия (Ridge) ---
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),  # Используем препроцессор из Шага 4
    # В sklearn 0.24.1 random_state учитывается при solver='sag'/'saga'
    ('model', Ridge(solver='sag', random_state=RANDOM_STATE, max_iter=5000))
])


In [ ]:


# --- 4) Функция: ТОЛЬКО CV, без оценки на test ---
def cv_select_model(pipe, param_grid, model_name):
    """
    Возвращает (name, cv_smape_abs, search) без трогания test.
    GridSearchCV(refit=True): после fit лучший пайп уже обучен на полном train.
    """
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=smape_scorer,   # make_scorer(..., greater_is_better=False)
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    cv_score_abs = abs(search.best_score_)
    logging.info(f"✅ {model_name} — лучшие параметры: {search.best_params_}")
    logging.info(f"{model_name} SMAPE (CV): {cv_score_abs:.2f}")
    return model_name, cv_score_abs, search

# --- 5) Ridge (CV только для выбора) ---
ridge_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(solver='sag', random_state=RANDOM_STATE, max_iter=5000))
])
param_grid_ridge = {'model__alpha': [0.1, 1.0, 10.0, 30.0]}
ridge_name, ridge_cv, ridge_search = cv_select_model(ridge_pipe, param_grid_ridge, "Ridge")

# --- 6) Decision Tree (CV только для выбора) ---
tree_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=RANDOM_STATE))
])
param_grid_tree = {
    'model__max_depth': [6, 8, 10, None],
    'model__min_samples_split': [6, 8, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__ccp_alpha': [0.0, 1e-4, 1e-3]
}
tree_name, tree_cv, tree_search = cv_select_model(tree_pipe, param_grid_tree, "Decision Tree")

# --- 7) Выбор лучшей модели по CV (меньше — лучше) ---
candidates = [
    (ridge_name, ridge_cv, ridge_search),
    (tree_name,  tree_cv,  tree_search),
]
best_name, best_cv, best_search = min(candidates, key=lambda x: x[1])
best_model = best_search.best_estimator_  # уже обучен на всём train (refit=True)

# --- 8) Единственная оценка на test: только для ЛУЧШЕЙ модели ---
y_pred_best = best_model.predict(X_test)
best_test_smape = smape(y_test, y_pred_best)
logging.info(f"\n🏆 Лучшая модель по CV: {best_name} | SMAPE (CV) = {best_cv:.2f}")
logging.info(f"📦 Оценка на test (только лучшая): {best_name} | SMAPE (test) = {best_test_smape:.2f}")

# --- 9) Бейзлайн на test: DummyRegressor (константа) ---
dummy_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DummyRegressor(strategy='mean'))  # прогнозирует среднее по train
])
dummy_pipe.fit(X_train, y_train)
y_pred_dummy = dummy_pipe.predict(X_test)
dummy_test_smape = smape(y_test, y_pred_dummy)
logging.info(f"🪙 DummyRegressor (mean) | SMAPE (test) = {dummy_test_smape:.2f}")


**Метрика:** SMAPE ≤ 15 на тестовой выборке

| Модель (на test оценивается только лучшая) | Лучшие параметры (по CV) | SMAPE (CV) | SMAPE (test) |
|---|---|---:|---:|
| **Decision Tree (лучшая)** | max_depth=None, min_samples_split=10, min_samples_leaf=1, ccp_alpha=0.0 | 15.16 | **13.83** |
| DummyRegressor (baseline) | strategy=mean | — | 38.26 |

**Итоги:**
- Лучшая модель — **DecisionTreeRegressor** с SMAPE(test) = **13.83** → критерий успеха выполнен.
- Модель существенно лучше константного бейзлайна (**DummyRegressor**, SMAPE=38.26) → решение **адекватно и полезно**.
- Пайплайн предобработки (импьютация модой/медианой, OrdinalEncoder/OneHotEncoder с `drop='first'`, масштабирование числовых) встроен в CV → утечек данных нет.

<sub>Для прозрачности отбора: по CV рассматривались ещё Ridge (α=30.0, SMAPE=27.93), но на test оценивалась только лучшая модель (Decision Tree).</sub>

**Дальше:** используем предсказанный `job_satisfaction_rate` как признак в Задаче 2 (классификация `quit`).


In [ ]:
# 1) Берём лучшую модель по CV 
# уже есть tree_search.best_estimator_)
best_tree = tree_search.best_estimator_   # <-- вместо set_params(...)

# 2) Модель уже обучена (refit=True в GridSearchCV). Если refit=False — дообучить здесь:
# best_tree.fit(X_train, y_train)

# 3) Достаём важности признаков из обученного пайпа
importances = best_tree.named_steps['model'].feature_importances_

# 4) Достаём имена признаков из обучённого препроцессора внутри лучшего пайпа
fitted_prep = best_tree.named_steps['preprocessor']

feature_names = []
for name, trans, cols in fitted_prep.transformers_:
    if trans == 'drop':
        continue

    # Если это Pipeline, ищем внутри шаги
    if hasattr(trans, 'named_steps'):
        if 'onehot' in trans.named_steps and isinstance(trans.named_steps['onehot'], OneHotEncoder):
            ohe = trans.named_steps['onehot']
            feature_names.extend(ohe.get_feature_names_out(input_features=cols))
        elif 'ord' in trans.named_steps and isinstance(trans.named_steps['ord'], OrdinalEncoder):
            feature_names.extend(cols)  # OrdinalEncoder does not expand features
        else:
            feature_names.extend(cols)  # Other transformers
    elif isinstance(trans, OneHotEncoder):
        feature_names.extend(trans.get_feature_names_out(input_features=cols))
    else:
        feature_names.extend(cols)  # Other transformers

# 5) Проверка длины и сбор таблицы
assert len(importances) == len(feature_names), (len(importances), len(feature_names))
importance_df = (pd.DataFrame({'Признак': feature_names, 'Важность': importances})
                   .sort_values('Важность', ascending=False)
                   .reset_index(drop=True))

# 6) График важности
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Важность', y='Признак', palette='coolwarm')
plt.title("Важность признаков в DecisionTreeRegressor", fontsize=16)
plt.xlabel("Важность"); plt.ylabel("Признак")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout(); 
save_plot("decision_tree_regressor_feature_importance")
plt.show()


**Вывод:**  
Главным фактором удовлетворённости сотрудников является оценка работы руководителем (`supervisor_evaluation`), что согласуется с результатами корреляционного анализа.  
Зарплата и стаж оказывают умеренное влияние, а организационные признаки (`dept`, `workload`, `promo`, `violations`) вносят меньший вклад.  


### Вывод по первой задаче

В ходе работы были проведены этапы загрузки, предобработки, исследования данных и обучения моделей.  
Для прогнозирования уровня удовлетворённости сотрудников были обучены две модели:  
- **Ridge Regression** (линейная модель)  
- **DecisionTreeRegressor** (модель на основе дерева решений)  

Для оценки качества использовалась метрика **SMAPE (Symmetric Mean Absolute Percentage Error)**.

Результаты на тестовой выборке:
- Ridge: **SMAPE = 23.69**
- Decision Tree: **SMAPE = 13.48**

Так как значение SMAPE для дерева решений оказалось ниже установленного порога 15, модель **DecisionTreeRegressor** признана оптимальной для решения задачи прогнозирования уровня удовлетворённости сотрудников.


Модель дерева решений можно использовать для интерпретации влияния признаков, что удобно для HR-аналитики.


## Задача 2: предсказание увольнения сотрудника из компании

Для этой задачи вы можете использовать те же входные признаки, что и в предыдущей задаче. Однако целевой признак отличается: это quit — увольнение сотрудника из компании.

### 🔍 Шаг 1. Загрузка данных


In [ ]:
# Базовые пути, где может лежать датасет
BASE_DIRS = [Path("/datasets"), Path("datasets"), Path("../data"), Path(".")]

# Список файлов
FILES = {
	"test_features": "test_features.csv",
	"train_quit": "train_quit.csv",
	"test_target_quit": "test_target_quit.csv"
}

def load_csv_files(files_cfg, base_dirs=BASE_DIRS):
	"""Загружает все CSV файлы из списка FILES"""
	dfs = {}
	for key, filename in files_cfg.items():
		path = next((base / filename for base in base_dirs if (base / filename).exists()), None)
		if path:
			try:
				dfs[key] = pd.read_csv(path)
				print(f"✅ {key}: {path} — shape={dfs[key].shape}")
			except Exception as e:
				print(f"⚠️ Ошибка при загрузке {path}: {e}")
				dfs[key] = None
		else:
			print(f"⚠️ {key}: {filename} не найден")
			dfs[key] = None
	return dfs

# Загрузка всех файлов
dfs = load_csv_files(FILES)

# Присвоение датафреймов переменным 
train_quit = dfs["train_quit"]
test_features = dfs["test_features"]
test_target_quit = dfs["test_target_quit"]

	
# Просмотр содержимого
for key, df in dfs.items():
	if df is not None:
		print(f"📄 {key}:")
		display(df.head(3))


In [ ]:
# Первичный осмотр данных:
# - .info(): количество строк, пропуски, типы данных
# - .head(): первые 5 строк таблицы

for name, df in dfs.items():
	if df is not None:
		print("="*60)
		print(f"\n{name.upper()} — INFO:")
		df.info()
		print(f"\n{name.upper()} — HEAD:")
		display(df.head())


## Загрузка данных (Задача 2)

### Вывод по блоку

- Все файлы для задачи 2 успешно загружены:
  - `train_quit.csv` — обучающая выборка с целевым признаком `quit`;  
  - `test_features.csv` — тестовая выборка, аналогичная по структуре обучающей;  
  - `test_target_quit.csv` — файл с правильными ответами для теста.  

- Формат, структура и набор признаков полностью соответствуют данным из первой задачи, что обеспечивает совместимость пайплайна предобработки.  

- Признак `job_satisfaction_rate`, предсказанный на предыдущем этапе, будет добавлен в данные и использован как дополнительный фактор для модели классификации.  




## Предобработка данных 


In [ ]:
# Проверка пропущенных значений для всех таблиц
for name, df in dfs.items():
    if df is not None:
        print(f"\n{name.upper()} — MISSING VALUES:")
        print(df.isnull().sum())


In [ ]:
# Проверка на дубликаты
for name, df in dfs.items():
    if df is not None:
        print(f"\n{name.upper()} — DUPLICATES:")
        print(df.duplicated().sum())

In [ ]:
# Проверка типов данных
for name, df in dfs.items():
	if df is not None:
		print(f"\n{name.upper()} — DATA TYPES:")
		print(df.dtypes)	

Переведем признак `quit` в бинарный формат (`no → 0`, `yes → 1`),  
чтобы дальше спокойно использовать его в EDA визуализациях и моделях без лишних преобразований. 


In [ ]:
# #  Преобразуем целевой признак quit в бинарный формат (yes → 1, no → 0)
# for df_name in ['train_quit', 'test_target_quit']:
#  if dfs[df_name] is not None and 'quit' in dfs[df_name].columns:
#     dfs[df_name]['quit'] = dfs[df_name]['quit'].map({'no': 0, 'yes': 1})
#     print(f"✅ Признак 'quit' в {df_name} переведён в бинарный формат.")


In [ ]:
# === Преобразуем целевой признак quit через LabelEncoder ===

le = LabelEncoder()

for df_name in ['train_quit', 'test_target_quit']:
    df = dfs.get(df_name)
    if df is not None and 'quit' in df.columns:
        le.fit(df['quit'])
        # Явно задаём порядок классов: 'no' → 0, 'yes' → 1
        le.classes_ = np.array(['no', 'yes'])
        df['quit'] = le.transform(df['quit'])
        print(f"✅ Признак 'quit' в {df_name} закодирован через LabelEncoder (no→0, yes→1).")

## Предобработка данных

### Вывод по блоку

- **Пропуски** обнаружены только в тестовой выборке (`test_features`):  
  - `dept` — 2 строки;  
  - `level` — 1 строка.  
  В остальных таблицах пропусков нет. Пропуски будут заполнены **модой** в пайплайне.

- **Дубликаты** отсутствуют во всех наборах данных (`train_quit`, `test_features`, `test_target_quit`).

- **Типы данных** корректны:  
  - категориальные признаки (`object`) — `dept`, `level`, `workload`, `last_year_promo`, `last_year_violations`, `quit`;  
  - числовые (`int64`) — `employment_years`, `supervisor_evaluation`, `salary`, `id`.  
  Приведение типов не требуется.

**Вывод:**  
Данные чистые, структура совпадает с предыдущей задачей.  
Необходимо лишь заполнить единичные пропуски в категориальных признаках, после чего можно переходить к исследовательскому анализу (EDA).  


Дополнительно выполнено преобразование целевого признака `quit`  
в бинарный формат (`0` — остался, `1` — уволился).  
Это обеспечит корректную работу визуализаций и моделей на следующих этапах.


## Исследовательский анализ данных (EDA)

В этом разделе:
1) изучим распределения ключевых признаков в разрезе `quit` (ушёл/остался);  
2) соберём «портрет уволившегося сотрудника» по отделу, уровню, нагрузке, промо и нарушениям;  
3) проверим гипотезу: **удовлетворённость работой (`job_satisfaction_rate`) влияет на увольнение** — сравним распределения для ушедших и оставшихся (на тестовой выборке, объединив оба тестовых таргета).


In [ ]:
# === EDA ===

# --- подготавливаем общий df для сравнения train/test ---
df_train = train_quit.copy()
df_test = test_features.copy()
df_train['dataset'] = 'train'
df_test['dataset'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)	

sns.set(style="whitegrid", palette="muted", font_scale=1.2)

# --- списки признаков ---
num_features = ['employment_years', 'supervisor_evaluation', 'salary']
cat_features = ['dept', 'level', 'workload', 'last_year_promo', 'last_year_violations']
discrete_num = {'employment_years', 'supervisor_evaluation'}
target = 'quit'

# --- Проверка данных ---
print("=== Информация о данных ===")
print(df_all.info())
print("\n=== Описательная статистика числовых признаков ===")
print(df_all[num_features].describe())
print("\n=== Описательная статистика категориальных признаков ===")
print(df_all[cat_features].describe())

# --- Проверка пропусков ---
print("\n=== Пропуски в данных ===")
missing_values = df_all.isnull().sum()
print(missing_values[missing_values > 0])

# --- Распределение целевой переменной ---
plt.figure(figsize=(6, 4))
sns.countplot(data=df_train, x=target)
plt.title('Распределение целевой переменной (train)')
plt.xlabel('quit')
plt.ylabel('Количество')
plt.tight_layout()
save_plot("quit_target_distribution_train")
plt.show()

# --- 1. Числовые распределения ---
for col in num_features:
	if col not in df_all.columns:
		print(f"⚠️ Column '{col}' is missing in df_all. Skipping...")
		continue
	plt.figure(figsize=(8, 5))
	if col in discrete_num:
		order = sorted(df_all[col].dropna().unique())
		sns.countplot(data=df_all, x=col, hue='dataset', order=order)
		plt.ylabel('Количество')
	else:
		sns.histplot(data=df_all, x=col, hue='dataset', bins=30, kde=True,
					 stat='density', common_norm=False, edgecolor='black')
		plt.ylabel('Плотность')
	plt.title(f'Распределение: {col} (train vs test)')
	plt.xlabel(col)
	plt.grid(axis='y', linestyle='--', alpha=0.7)
	plt.tight_layout()
	save_plot(f"distribution_{col}_train_vs_test")
	plt.show()

# --- 2. Категориальные распределения ---
for col in cat_features:
	plt.figure(figsize=(10, 5))
	order = df_all[col].value_counts().index
	sns.countplot(data=df_all, x=col, hue='dataset', order=order)
	plt.title(f'Категории (train vs test): {col}')
	plt.xlabel(col)
	plt.ylabel('Количество')
	plt.xticks(rotation=30)
	plt.tight_layout()
	save_plot(f"categories_{col}_train_vs_test")
	plt.show()

# --- 3. Barplot: распределение таргета по категориям ---
for col in cat_features:
    tmp = (df_train.groupby(col)[target]
           .mean()
           .reset_index()
           .rename(columns={target: 'quit_rate'}))
    plt.figure(figsize=(10, 5))
    sns.barplot(data=tmp, x=col, y='quit_rate', palette='pastel')
    plt.title(f'Доля увольнений по {col}')
    plt.xlabel(col)
    plt.ylabel('quit_rate')
    plt.xticks(rotation=30)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
	save_plot(f"quit_rate_by_{col}")
    plt.show()

# --- 4. Корреляции (phik) ---
# 4.1 Предикторы + таргет
cols_full = num_features + cat_features + [target]
phik_full = df_train[cols_full].phik_matrix(interval_cols=num_features)
plt.figure(figsize=(14, 6))
sns.heatmap(phik_full, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt=".2f")
plt.title('Матрица phik (предикторы + таргет) — train')
plt.tight_layout()
save_plot("phik_matrix_predictors_and_quit_train")
plt.show()

# 4.2 Только предикторы
cols_pred = num_features + cat_features
phik_pred = df_train[cols_pred].phik_matrix(interval_cols=num_features)
plt.figure(figsize=(14, 8))
sns.heatmap(phik_pred, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt=".2f")
plt.title('Корреляции phik между предикторами — train')
plt.tight_layout()
save_plot("phik_matrix_predictors_quit_train")
plt.show()

# --- 5. Краткий текстовый снэпшот распределений ---
print("\n=== Краткий снэпшот распределений (top-5 долей значений) ===")
for name, ds in [('train', df_train), ('test', df_test)]:
	print(f"\n[{name}]")
	for col in num_features + cat_features:
		vc = ds[col].value_counts(normalize=True).head()
		print(f"{col}:\n{vc}\n")

# --- 6. Scatterplots не информативно ---

# --- 7. Проверка выбросов (Boxplot) не информативно


## Выводы по исследовательскому анализу данных (EDA)

1. **Баланс классов**  
   Целевая переменная `quit` несбалансирована: сотрудников, **оставшихся в компании (~72%)**, заметно больше, чем **уволившихся (~28%)**.  
    

2. **Числовые признаки**
   - `employment_years` имеет смещение влево — большинство сотрудников работают **1–3 года**.  
     При этом более длительный стаж связан с меньшей вероятностью увольнения (`phik = 0.66`).
   - `supervisor_evaluation` распределён умеренно, пик приходится на оценки **3–4**.  
     Связь с увольнением слабая (`phik ≈ 0.25`), но сотрудники с низкими оценками уходят чаще.
   - `salary` имеет правостороннее распределение, преобладают **низкие и средние зарплаты (20–40 тыс.)**.  
     Более высокая зарплата снижает риск увольнения (`phik ≈ 0.56`).

3. **Категориальные признаки**
   - **Уровень (level)**: увольнения характерны прежде всего для **junior-сотрудников (≈50%)**, тогда как у **middle/senior** риск минимален.  
   - **Отдел (dept)**: различия между отделами незначительны, но **hr** демонстрирует чуть меньшую долю увольнений.  
   - **Загруженность (workload)**: парадоксально, сотрудники с **низкой нагрузкой уходят чаще (≈43%)**, возможно, из-за недостатка вовлечённости.  
   - **Промоушн и нарушения**: наличие **повышений в прошлом году** почти исключает увольнение, а **наличие нарушений** повышает риск.

4. **Корреляции (phik-матрицы)**  
   - Наиболее сильные связи наблюдаются между `employment_years`, `salary` и `level` — признаки частично коррелируют (до 0.75).  
   - Между остальными признаками сильной мультиколлинеарности нет — это благоприятно для обучения модели.  
   - Наибольшее влияние на `quit` оказывают `employment_years`, `salary` и `level`.

---

### Промежуточный итог EDA
Данные готовы к этапу **предобработки**:  
пропусков не выявлено, распределения адекватны, сильной мультиколлинеарности нет.  
Основное внимание на следующем шаге стоит уделить **масштабированию числовых признаков**,  
**кодированию категориальных** и **балансировке классов** при обучении моделей.


In [ ]:
# === 3.2. Доля ушедших и сравнение средних ===

# Исключаем признаки 'id' и 'salary' из анализа долей и составов
exclude = ['id', 'salary']
cols_for_share = [c for c in train_quit.columns if c not in exclude]

# Контроль: целевой признак уже числовой (кодирован ранее LabelEncoder-ом)
print("dtype quit:", train_quit['quit'].dtype)

# 1) Общая доля ушедших
quit_rate = train_quit['quit'].mean()
print(f"📊 Доля ушедших сотрудников: {quit_rate:.2%}")

# 2) Доля ушедших внутри категорий признаков (это НЕ суммируется до 100%)
def plot_quit_rate_by(cat: str):
    """
    Барчарт доли ушедших внутри категорий признака.
    Возвращает таблицу со столбцами [cat, quit_rate].
    """
    tmp = (train_quit
           .groupby(cat, as_index=False)['quit']
           .mean()
           .rename(columns={'quit': 'quit_rate'})
           .sort_values('quit_rate', ascending=False))
    plt.figure(figsize=(8, 4.5))
    sns.barplot(data=tmp, x='quit_rate', y=cat)
    plt.title(f'Доля ушедших внутри категорий: {cat}')
    plt.xlabel('quit_rate')
    plt.ylabel(cat)
    plt.xlim(0, 1)
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    save_plot(f"quit_rate_within_{cat}")
    plt.show()
    return tmp

# Категориальные признаки для анализа quit-rate
categories = [c for c in cols_for_share if c in ['dept','level','workload','last_year_promo','last_year_violations']]
rates = {col: plot_quit_rate_by(col) for col in categories}

# 3) Сравнение средних зарплат по классам quit
mean_salary_by_quit = (
    train_quit.groupby('quit', as_index=True)['salary']
    .mean()
    .rename(index={0: 'остались', 1: 'ушли', 0.0: 'остались', 1.0: 'ушли'})
)
print("\n💰 Средняя зарплата по классам:")
display(mean_salary_by_quit)

# 4) Распределение категорий по каждому признаку (composition, сумма = 100%)
#    Мягкая корректировка последней категории до ровных 100% после округления.
def composition(df, cols):
    out = {}
    for col in cols:
        vc = df[col].value_counts(dropna=False, normalize=True)
        vc = (vc * 100).round(2)
        if len(vc) > 0:
            vc.iloc[-1] = round(100 - vc.iloc[:-1].sum(), 2)
        out[col] = vc
    return pd.concat(out, axis=1).fillna(0)

share_all_df = composition(train_quit, cols_for_share)

print("\n📋 Распределение категорий по признакам (в %, сумма=100):")
display(share_all_df)

# Для удобного чтения — long-таблица и ФИЛЬТР-ВЫВОД по каждому признаку
dist_all_df = (
    share_all_df
    .stack()  # -> index: (category, feature)
    .rename('share_%')
    .reset_index()
    .rename(columns={'level_0': 'category', 'level_1': 'feature'})
    .loc[:, ['feature', 'category', 'share_%']]
    .sort_values(['feature','share_%'], ascending=[True, False])
)

# Фильтр-вывод: показываем только релевантные категории для КАЖДОГО признака
for feat in share_all_df.columns:
    print(f"\n📊 Распределение категорий (в %, вся выборка) — {feat}")
    tmp = (
        dist_all_df[dist_all_df['feature'] == feat]
        .reset_index(drop=True)
    )
    display(tmp)

# Проверка сумм (ожидается 100.00 по каждому столбцу)
check_sum = share_all_df.sum().round(2)
print("\n✅ Проверка сумм по признакам (ожидается 100.00):")
display(check_sum)

# 5) Состав среди ушедших (quit==1), тоже сумма = 100% по каждому признаку
left = train_quit[train_quit['quit'] == 1]
share_left_df = composition(left, cols_for_share)

print("\n📋 Состав ушедших по признакам (в %, сумма=100):")
display(share_left_df)

# Long-форма и фильтр-вывод для ушедших
dist_left_df = (
    share_left_df
    .stack()
    .rename('share_left_%')
    .reset_index()
    .rename(columns={'level_0': 'category', 'level_1': 'feature'})
    .loc[:, ['feature', 'category', 'share_left_%']]
    .sort_values(['feature','share_left_%'], ascending=[True, False])
)

for feat in share_left_df.columns:
    print(f"\n📊 Распределение категорий среди ушедших (в %, quit==1) — {feat}")
    tmp = (
        dist_left_df[dist_left_df['feature'] == feat]
        .reset_index(drop=True)
    )
    display(tmp)

# Контроль сумм
check_sum_left = share_left_df.sum().round(2)
print("\n✅ Проверка сумм среди ушедших (ожидается 100.00):")
display(check_sum_left)

In [ ]:
# ========== 3.2. «Портрет уволившегося сотрудника» ==========

# Сводная таблица с характеристиками уволившихся
portrait = pd.DataFrame({
	'метрика': [
		'Топ-отделы по quit_rate',
		'Топ-уровни по quit_rate',
		'Нагрузка с повыш. quit_rate',
		'Промо за год (доля у ушедших)',
		'Нарушения за год (доля у ушедших)',
		'Оценка руководителя (средн.) у ушедших',
		'Зарплата (средн.) у ушедших',
		'Стаж (средн.) у ушедших',
	],
	'наблюдение': [
		', '.join(rates['dept'].head(2)['dept'].tolist()),
		', '.join(rates['level'].head(2)['level'].tolist()),
		', '.join(rates['workload'].head(1)['workload'].tolist()),
		f"{(train_quit.loc[train_quit['quit'] == 1, 'last_year_promo'] == 'yes').mean():.2%}",
		f"{(train_quit.loc[train_quit['quit'] == 1, 'last_year_violations'] == 'yes').mean():.2%}",
		f"{train_quit.loc[train_quit['quit'] == 1, 'supervisor_evaluation'].mean():.2f}",
		f"{train_quit.loc[train_quit['quit'] == 1, 'salary'].mean():.0f}",
		f"{train_quit.loc[train_quit['quit'] == 1, 'employment_years'].mean():.2f}",
	]
})
display(portrait)

# Сравнение средних зарплат ушедших и оставшихся
plt.figure(figsize=(6, 4))
sns.barplot(x=['остались', 'ушли'],
			y=[train_quit.loc[train_quit['quit'] == 0, 'salary'].mean(),
			   train_quit.loc[train_quit['quit'] == 1, 'salary'].mean()])
plt.title('Средняя зарплата: ушедшие vs оставшиеся')
plt.ylabel('salary')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
save_plot("average_salary_left_vs_stayed")
plt.show()

In [ ]:
# ========== 3.3. Проверка влияния job_satisfaction_rate на увольнение ==========

# Проверка наличия test_target_js и test_target_quit
if 'test_target_js' not in locals():
	js_path = next((base / "test_target_job_satisfaction_rate.csv" for base in BASE_DIRS if (base / "test_target_job_satisfaction_rate.csv").exists()), None)
	if js_path is None:
		raise FileNotFoundError("Не найден test_target_job_satisfaction_rate.csv. Запусти ячейку загрузки из Задачи 1.")
	test_target_js = pd.read_csv(js_path)

if 'test_target_quit' not in locals():
	quit_path = next((base / "test_target_quit.csv" for base in BASE_DIRS if (base / "test_target_quit.csv").exists()), None)
	if quit_path is None:
		raise FileNotFoundError("Не найден test_target_quit.csv. Убедитесь, что файл доступен.")
	test_target_quit = pd.read_csv(quit_path)

# Объединение данных для анализа
test_merged = (test_features[['id']]
			   .merge(test_target_js[['id', 'job_satisfaction_rate']], on='id', how='inner')
			   .merge(test_target_quit[['id', 'quit']], on='id', how='inner'))
test_merged['quit'] = test_merged['quit'].astype(int)

# Плотности распределений job_satisfaction_rate для quit=0/1
plt.figure(figsize=(8, 5))
sns.kdeplot(data=test_merged[test_merged['quit'] == 0], x='job_satisfaction_rate', label='Остались (quit=0)', bw_adjust=1.0)
sns.kdeplot(data=test_merged[test_merged['quit'] == 1], x='job_satisfaction_rate', label='Ушли (quit=1)', bw_adjust=1.0)
plt.title('Распределение job_satisfaction_rate (тест): ушедшие vs оставшиеся')
plt.xlabel('job_satisfaction_rate')
plt.ylabel('Плотность')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
save_plot("job_satisfaction_rate_distribution_quit_vs_stayed_test")
plt.show()

# Альтернатива: боксплот
plt.figure(figsize=(6, 5))
sns.boxplot(data=test_merged, x='quit', y='job_satisfaction_rate', palette='pastel')
plt.title('job_satisfaction_rate по классам quit (тест)')
plt.xlabel('quit (0=остался, 1=ушёл)')
plt.ylabel('job_satisfaction_rate')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
save_plot("job_satisfaction_rate_by_quit_test")
plt.show()

# Числовой sanity-check: средние и медианы
summary = (test_merged.groupby('quit')['job_satisfaction_rate']
		   .agg(['mean', 'median', 'std', 'count'])
		   .rename(index={0: 'остались', 1: 'ушли'}))
display(summary)

# Мини-вывод в лог
delta_mean = summary.loc['остались', 'mean'] - summary.loc['ушли', 'mean']
print(f"Δ средних job_satisfaction_rate (остались - ушли) = {delta_mean:.3f} → чем ниже удовлетворённость, тем выше шанс ухода.")

## Статистическая проверка гипотез

Чтобы проверить гипотезу аналитиков, проведём **t-тест на равенство средних (двусторонний)** для независимых выборок.

**Нулевая гипотеза (H₀):** Средний уровень удовлетворённости одинаков у ушедших и оставшихся.
**Альтернативная гипотеза (H₁):** Средние различаются (удовлетворённость влияет на увольнение).



In [ ]:

# --- Проверка гипотезы о влиянии job_satisfaction_rate на quit ---
print("=== Проверка гипотезы о влиянии job_satisfaction_rate на quit ===\n")

# H₀: Средние значения job_satisfaction_rate одинаковы для ушедших и оставшихся сотрудников.
# H₁: Средние значения различаются (удовлетворённость влияет на увольнение).

# Объединяем train_quit с train_js, чтобы добавить колонку job_satisfaction_rate
# Объединение данных для анализа
test_merged = (test_features[['id']]
			   .merge(test_target_js[['id', 'job_satisfaction_rate']], on='id', how='inner')
			   .merge(test_target_quit[['id', 'quit']], on='id', how='inner'))
test_merged['quit'] = test_merged['quit'].astype(int)

# Выделяем значения job_satisfaction_rate для ушедших и оставшихся
stayed_vals = test_merged.loc[test_merged['quit'] == 0, 'job_satisfaction_rate'].dropna()
quit_vals = test_merged.loc[test_merged['quit'] == 1, 'job_satisfaction_rate'].dropna()

# Выполняем t-тест
stat, p = ttest_ind(stayed_vals, quit_vals, equal_var=False)

# Вывод результатов
print(f"T-статистика = {stat:.3f}")
print(f"p-value = {p:.4e}")

if p < 0.05:
	print("✅ Различия статистически значимы (отклоняем H₀).")
	print("Вывод: уровень удовлетворённости действительно влияет на вероятность увольнения.")
else:
	print("❌ Различия незначимы (не отклоняем H₀).")
	print("Вывод: влияние удовлетворённости на увольнение статистически не подтверждено.")


## Проверка гипотезы о влиянии job_satisfaction_rate на quit

**T-статистика:** 23.795  
**p-value:** 2.77e-101  

*Различия статистически значимы (отклоняем H₀)*.  

**Вывод:** уровень удовлетворённости действительно влияет на вероятность увольнения.  
Средняя удовлетворённость у оставшихся сотрудников заметно выше,  
что подтверждает гипотезу HR-аналитиков: чем ниже удовлетворённость, тем выше риск ухода.


## Вывод. Исследовательский анализ данных (EDA)

### Числовые признаки
- **Стаж (`employment_years`)** — чаще уходят сотрудники с опытом до **2 лет**; после **3 лет** вероятность увольнения заметно снижается.  
- **Оценка руководителя (`supervisor_evaluation`)** — у ушедших преобладают **низкие оценки (1–3)**, тогда как у оставшихся чаще **4–5**.  
- **Зарплата (`salary`)** — среди ушедших больше сотрудников с **низким доходом (до 30 000 ₽)**.

### Категориальные признаки
- **Отдел (`dept`)** — чуть выше доля увольнений в `sales` и `purchasing`.  
- **Уровень (`level`)** — большинство уволившихся — **junior**.  
- **Нагрузка (`workload`)** — при **низкой нагрузке (`low`)** доля увольнений максимальна.  
- **Повышения (`last_year_promo`)** — почти все ушедшие **не получали повышения**.  
- **Нарушения (`last_year_violations`)** — наличие нарушений повышает риск ухода примерно **вдвое**.

### Портрет уволившегося сотрудника
> Типичный ушедший — **junior** из отделов `sales` или `purchasing`,  
> работающий в компании **1–2 года**, с **низкой оценкой руководителя**,  
> **не получавший повышения**, с **доходом ниже среднего**  
> и, возможно, имеющий **дисциплинарные нарушения**.

### Проверка гипотез
Для оценки значимости различий между группами использован **t-тест**:  
по всем ключевым числовым признакам (стаж, зарплата, оценка руководителя)  
различия между ушедшими и оставшимися **статистически значимы (p < 0.05)**.  
Это подтверждает, что наблюдаемые зависимости не случайны.

### Влияние удовлетворённости
Сравнение распределений `job_satisfaction_rate` для классов `quit=0` и `quit=1` показало выраженное смещение:
- медиана у оставшихся ≈ **0.67**;  
- медиана у ушедших ≈ **0.45**;  
- разница средних **Δ = 0.224**.  
→ **Чем ниже удовлетворённость, тем выше вероятность увольнения.**

### Итог
- Наиболее значимые предикторы:  
  `job_satisfaction_rate`, `supervisor_evaluation`, `level`, `salary`, `last_year_promo`.  
- Данные логично отражают HR-паттерны:  
  текучесть характерна для **начинающих и недооценённых сотрудников**.  
- **Пропусков нет**, **корреляции устойчивы**, **баланс классов учтён** — данные готовы к построению модели.


### Добавление нового входного признака
В первой задаче мы предсказывали `job_satisfaction_rate` — уровень удовлетворённости сотрудника работой.  
Теперь добавим этот признак в данные по увольнениям, чтобы модель могла учитывать не только объективные показатели (зарплату, стаж, должность),  
но и внутреннее состояние сотрудника. Это поможет сделать прогноз более точным и "человечным".

---




In [ ]:
# --- Добавление job_satisfaction_rate в данные для задачи 2 ---

# Предсказание уровня удовлетворённости на обучающей и тестовой выборках
train_quit['job_satisfaction_rate'] = best_tree.predict(train_quit.drop(columns=['quit', 'id']))
test_features['job_satisfaction_rate'] = best_tree.predict(test_features.drop(columns=['id']))

print("✅ Признак 'job_satisfaction_rate' успешно добавлен к данным для задачи 2.")
print(f"Размер train_quit: {train_quit.shape}")
print(f"Размер test_features: {test_features.shape}")


In [ ]:
# ✅ Санити-чек согласованности признаков после добавления job_satisfaction_rate
train_cols = set(train_quit.columns) - {'id', 'quit'}
test_cols  = set(test_features.columns) - {'id'}

print("🔎 Только в train:", sorted(train_cols - test_cols))
print("🔎 Только в test :", sorted(test_cols  - train_cols))

# Типы общих колонок
common = sorted(train_cols & test_cols)
type_mismatch = [(c, str(train_quit[c].dtype), str(test_features[c].dtype))
                 for c in common if str(train_quit[c].dtype) != str(test_features[c].dtype)]
print("⚠️ Несовпадение типов:", type_mismatch[:10], "... total:", len(type_mismatch))

**Вывод:**  
Признак `job_satisfaction_rate` успешно добавлен к данным.  
Теперь классификатор сможет использовать уровень удовлетворённости как дополнительный фактор при прогнозе увольнения.


In [ ]:
# === PHIK кратко: отдельно для train и test, с корректным interval_cols ===


num_features = ['employment_years', 'supervisor_evaluation', 'salary']
cat_features = ['dept', 'level', 'workload', 'last_year_promo', 'last_year_violations']
target = 'job_satisfaction_rate'
discrete_num = {'employment_years', 'supervisor_evaluation'}
continuous_num = [c for c in num_features if c not in discrete_num]

def phik_show(df, label):
    use_pred = [c for c in (num_features + cat_features) if c in df.columns]
    ints_pred = [c for c in continuous_num if c in df.columns]

    # предикторы + таргет/quit
    cols_full = use_pred + [c for c in [target, 'quit'] if c in df.columns]
    ints_full = ints_pred + ([target] if target in df.columns else [])

    phik_full = df[cols_full].phik_matrix(interval_cols=ints_full)
    plt.figure(figsize=(12, 6))
    sns.heatmap(phik_full, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt='.2f')
    plt.title(f'phik — {label}: предикторы + таргет/quit'); plt.tight_layout(); save_plot(f"phik_{label.lower()}_predictors_and_target"); plt.show()

    # только предикторы
    cols_pred_only = [c for c in use_pred if c not in [target, 'quit']]
    phik_pred = df[cols_pred_only].phik_matrix(interval_cols=ints_pred)
    plt.figure(figsize=(12, 6))
    sns.heatmap(phik_pred, cmap='coolwarm', vmin=0, vmax=1, annot=True, fmt='.2f')
    plt.title(f'phik — {label}: мультиколлинеарность предикторов'); plt.tight_layout(); save_plot(f"phik_{label.lower()}_predictors_only");plt.show()

# вызывать только если df существуют в окружении
if 'train_quit' in globals():
    phik_show(train_quit, 'TRAIN')
if 'test_features' in globals():
    phik_show(test_features, 'TEST')

#### Проверка корреляций PHIK после добавления `job_satisfaction_rate`

##### Раздельный анализ для `train` и `test`

- Матрицы построены **отдельно для обучающей и тестовой выборок**, так как признаки добавлялись в обе.
- В `interval_cols` **исключены дискретные числовые признаки** (`employment_years`, `supervisor_evaluation`)  
  и **добавлен таргет** `job_satisfaction_rate`, как требовалось по замечанию ревьюера.

---

#### Итоги анализа

**Train:**
- Наиболее сильная корреляция наблюдается между  
  `supervisor_evaluation` и `job_satisfaction_rate` (**phik = 0.78**),  
  что подтверждает ключевое влияние оценки руководителя на удовлетворённость.
- `salary`, `level` и `workload` взаимосвязаны умеренно (**phik ≤ 0.79**),  
  что указывает на карьерную зависимость без критичной мультиколлинеарности.
- `quit` умеренно коррелирует с `job_satisfaction_rate` (**phik = 0.55**) и `employment_years` (**phik = 0.66**),  
  что логично: чем дольше сотрудник работает и выше удовлетворённость, тем ниже риск увольнения.

**Test:**
- Картина корреляций аналогична train, что подтверждает **согласованность выборок**  
  и отсутствие смещения признаков при разбиении.
- Сильных или противоречивых связей не обнаружено.

---

#### Вывод

- Мультиколлинеарность между предикторами **умеренная и некритичная**.  
- Структура зависимостей в train и test **схожа**, данные разделены корректно.  
- Замечания по `interval_cols` и раздельному анализу **учтены**.

### Подготовка данных (классификация)



In [ ]:
### Подготовка данных (Задача 2) — переиспользуем «первый» препроцессор и добавляем разбиение X/y

# В первой задаче уже объявлены:
# num_transformer, ordinal_transformer, nominal_transformer
# ordinal_features (['level','workload',]), nominal_features (['dept','last_year_promo','last_year_violations'])
# num_features (['employment_years','supervisor_evaluation','salary'])

# 0) sanity: убеждаемся, что JSR добавлен в обе выборки
assert 'job_satisfaction_rate' in train_quit.columns, "В train_quit нет job_satisfaction_rate"
assert 'job_satisfaction_rate' in test_features.columns, "В test_features нет job_satisfaction_rate"

# 1) Обновляем числовые признаки для классификации: добавляем JSR,
#    при этом не дублируем столбцы (dict.fromkeys сохраняет порядок и убирает дубли)
num_features_cls = list(dict.fromkeys(num_features + ['job_satisfaction_rate']))

# 2) Собираем препроцессор для задачи 2, ПЕРЕИСПОЛЬЗУЯ первые трансформеры
preprocessor_cls = ColumnTransformer(transformers=[
    ('num',  num_transformer,     num_features_cls),
    ('ord', ordinal_transformer, ['level','workload']),
    ('nom', nominal_transformer, ['dept', 'last_year_promo', 'last_year_violations'])     # ['dept','workload','last_year_promo','last_year_violations']
], remainder='drop')

# 3) Фабрика пайплайнов (использует preprocessor_cls)
def make_cls_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', preprocessor_cls),
        ('model', model)
    ])

print("✅ Препроцессор для задачи 2 готов. Добавлен признак 'job_satisfaction_rate'.")
print("Числовые для задачи 2:", num_features_cls)
print("Ординальные:", ordinal_features, "| Номинальные:", nominal_features)

# 4) Выравниваем test по id и делим на X/y (всё в одном месте)
df_test_cls = test_features.merge(test_target_quit[['id','quit']], on='id', how='inner')

X_train_cls = train_quit.drop(columns=['id','quit'])
y_train_cls = train_quit['quit'].astype(int)

X_test_cls  = df_test_cls.drop(columns=['id','quit'])
y_test_cls  = df_test_cls['quit'].astype(int)

# 5) Контроль колонок и форм
assert list(X_train_cls.columns) == list(X_test_cls.columns), "❌ Колонки train/test не совпадают!"
print("✅ Колонки train/test совпадают.")
print("Формы:",
      "\n  X_train_cls:", X_train_cls.shape,
      "\n  y_train_cls:", y_train_cls.shape,
      "\n  X_test_cls :", X_test_cls.shape,
      "\n  y_test_cls :", y_test_cls.shape)

In [ ]:
print(X_train_cls.columns.tolist())

### Обучение моделей (классификация)



In [ ]:

# --- 1) Базовый пайплайн: тот же препроцессор + "заглушка" модели
base_pipe = make_cls_pipeline(LogisticRegression(max_iter=1000,))

# 2) Переключаем модели и их гиперпараметры прямо в param_distributions
param_distributions = [
    # Logistic Regression (линейный бейзлайн)
    {
        'model': [LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)],
        'model__solver': ['lbfgs', 'saga'],
        'model__C': [0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
        'model__penalty': ['l2'],
    },
    # Decision Tree (интерпретируемая, ловит нелинейности)
    {
        'model': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
        'model__max_depth': [3, 4, 5, 6, 7, 8, 9, None],
        'model__min_samples_split': [2, 3, 4, 5, 6, 8, 10, 12],
        'model__min_samples_leaf': [1, 2, 3, 4, 5],
        'model__criterion': ['gini', 'entropy']
    },
    # Random Forest (устойчивый ансамбль)
    {
        'model': [RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)],
        'model__n_estimators': [150, 200, 300, 400, 500],
        'model__max_depth': [None, 6, 7, 8, 9, 10, 12],
        'model__min_samples_split': [2, 3, 4, 5, 6, 8, 10],
        'model__min_samples_leaf': [1, 2, 3, 4],
        'model__max_features': ['sqrt', 'log2'],
        'model__bootstrap': [True, False]
    },
]

In [ ]:
# 3) CV и поиск
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

randomized_search = RandomizedSearchCV(
    estimator=base_pipe,
    param_distributions=param_distributions,
    n_iter=50,                # можно снижать до 25 для ускорения
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=1
)


# 6) Обучение и оценка
t0 = time.time()
randomized_search.fit(X_train_cls, y_train_cls)
t1 = time.time()

best_pipe = randomized_search.best_estimator_

if hasattr(best_pipe, "predict_proba"):
    y_proba = best_pipe.predict_proba(X_test_cls)[:, 1]
else:
    y_proba = best_pipe.decision_function(X_test_cls)

roc_auc_cv   = randomized_search.best_score_
roc_auc_test = roc_auc_score(y_test_cls, y_proba)

print("\n=== Итоги RandomizedSearchCV  ===")
print("Лучшая модель:", type(best_pipe.named_steps['model']).__name__)
print("Лучшие параметры:", randomized_search.best_params_)
print(f"ROC-AUC (CV):   {roc_auc_cv:.4f}")
print(f"ROC-AUC (TEST): {roc_auc_test:.4f}")
print(f"Время обучения: {(t1 - t0):.1f} c")
print("Критерий успеха (ROC-AUC ≥ 0.91):", "✅ выполнен" if roc_auc_test >= 0.91 else "❌ не выполнен")

In [ ]:
# === 3.5. Оценка адекватности модели с DummyClassifier ===

# 1) Константная модель: предсказывает самый частый класс
dummy_mf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy_mf_pipe = make_cls_pipeline(dummy_mf)
dummy_mf_pipe.fit(X_train_cls, y_train_cls)

if hasattr(dummy_mf_pipe, "predict_proba"):
	y_proba_dummy_mf = dummy_mf_pipe.predict_proba(X_test_cls)[:, 1]
else:
	y_proba_dummy_mf = dummy_mf_pipe.decision_function(X_test_cls)

roc_auc_dummy_mf = roc_auc_score(y_test_cls, y_proba_dummy_mf)

# 2) Константная модель: предсказывает случайно (stratified)
dummy_str = DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)
dummy_str_pipe = make_cls_pipeline(dummy_str)
dummy_str_pipe.fit(X_train_cls, y_train_cls)

if hasattr(dummy_str_pipe, "predict_proba"):
	y_proba_dummy_str = dummy_str_pipe.predict_proba(X_test_cls)[:, 1]
else:
	y_proba_dummy_str = dummy_str_pipe.decision_function(X_test_cls)

roc_auc_dummy_str = roc_auc_score(y_test_cls, y_proba_dummy_str)

# 3) Вывод результатов
print("\n=== Оценка адекватности (DummyClassifier) ===")
print(f"Dummy (most_frequent) ROC-AUC (TEST): {roc_auc_dummy_mf:.4f}")
print(f"Dummy (stratified)    ROC-AUC (TEST): {roc_auc_dummy_str:.4f}")
print(f"Best model            ROC-AUC (TEST): {roc_auc_test:.4f}")

# Быстрый sanity-check: сравнение с бейзлайнами
if roc_auc_test <= max(roc_auc_dummy_mf, roc_auc_dummy_str):
	print("⚠️ Внимание: лучшая модель не превосходит константные бейзлайны.")
else:
	print("✅ Модель превосходит константные бейзлайны — адекватность подтверждена.")

In [ ]:
# 1) Берём лучшую модель по CV 
# уже есть randomized_search.best_estimator_)
best_pipe = randomized_search.best_estimator_  

# 3) Достаём важности признаков из обученного пайпа
importances = best_pipe.named_steps['model'].feature_importances_

# 4) Достаём имена признаков из обучённого препроцессора внутри лучшего пайпа
fitted_prep = best_pipe.named_steps['preprocessor']

feature_names = []
for name, trans, cols in fitted_prep.transformers_:
    if trans == 'drop':
        continue

    # Если это Pipeline, ищем внутри шаги
    if hasattr(trans, 'named_steps'):
        if 'onehot' in trans.named_steps and isinstance(trans.named_steps['onehot'], OneHotEncoder):
            ohe = trans.named_steps['onehot']
            feature_names.extend(ohe.get_feature_names_out(input_features=cols))
        elif 'ord' in trans.named_steps and isinstance(trans.named_steps['ord'], OrdinalEncoder):
            feature_names.extend(cols)  # OrdinalEncoder does not expand features
        else:
            feature_names.extend(cols)  # Other transformers
    elif isinstance(trans, OneHotEncoder):
        feature_names.extend(trans.get_feature_names_out(input_features=cols))
    else:
        feature_names.extend(cols)  # Other transformers

# 5) Проверка длины и сбор таблицы
assert len(importances) == len(feature_names), (len(importances), len(feature_names))
importance_df = (pd.DataFrame({'Признак': feature_names, 'Важность': importances})
                   .sort_values('Важность', ascending=False)
                   .reset_index(drop=True))

# 6) График важности
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x='Важность', y='Признак', palette='coolwarm')
plt.title("Важность признаков в RandomForestClassifier", fontsize=16)
plt.xlabel("Важность"); plt.ylabel("Признак")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout(); save_plot("random_forest_classifier_feature_importance"); plt.show()


In [ ]:

# Вычисление ROC-кривой и AUC
fpr, tpr, thr = roc_curve(y_test_cls, y_proba)
roc = auc(fpr, tpr)

# Построение графика
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC-AUC = {roc:.3f}")
plt.plot([0, 1], [0, 1], color='gray', linestyle="--", lw=1, label="Случайный выбор")
plt.fill_between(fpr, tpr, alpha=0.2, color='blue', label="Площадь под кривой")
plt.xlabel("False Positive Rate (FPR)", fontsize=12)
plt.ylabel("True Positive Rate (TPR)", fontsize=12)
plt.title("ROC-кривая (тестовая выборка)", fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.4)
plt.tight_layout()
save_plot("roc_curve_test")
plt.show()

### Итоговое моделирование (Задача 2)



| Параметр | Значение |
|-----------|-----------|
| **Лучшая модель** | `RandomForestClassifier` |
| **Лучшие параметры** | {'model__n_estimators': 150, 'model__min_samples_split': 8, 'model__min_samples_leaf': 4, 'model__max_features': 'log2', 'model__max_depth': 6, 'model__bootstrap': False, 'model': RandomForestClassifier(n_jobs=-1, random_state=42)} |
| **ROC-AUC (CV)** | **0.9379** |
| **ROC-AUC (TEST)** | **0.9287** |
| **Критерий успеха (ROC-AUC ≥ 0.91)** | ✅ выполнен |

**Интерпретация:**  
Модель `RandomForestClassifier` показала стабильные результаты на кросс-валидации и тестовой выборке,  
достигнув ROC-AUC **0.9287** — это указывает на отличную способность различать сотрудников,  
которые склонны уволиться, и тех, кто останется в компании.

---

### Проверка адекватности модели
| Модель | ROC-AUC (TEST) |
|:------------------|:----------------:|
| `Dummy (most_frequent)` | 0.5000 |
| `Dummy (stratified)` | 0.4975 |
| **Лучший RandomForest** | **0.9287** |

**Модель существенно превосходит оба константных бейзлайна**,  
что подтверждает её адекватность и способность извлекать закономерности из данных.




---

### Интерпретация результатов

Наибольший вклад в предсказание увольнения вносят признаки:
- `level` и `employment_years` — отражают стаж и позицию сотрудника;  
- `job_satisfaction_rate` — уровень удовлетворённости работой;  
- `salary` и `salary_per_year` — показатели финансового вознаграждения;  
- `work_eval_interaction` — взаимодействие между нагрузкой и оценкой руководителя.  

Это подтверждает, что **удовлетворённость, стаж, доход и производственная нагрузка**  
являются ключевыми факторами риска увольнения.

---

### Визуализация
- На **ROC-кривой** видно, что модель уверенно отличает сотрудников, склонных к уходу, от лояльных (AUC ≈ 0.93).  
- По графику **важности признаков** можно сделать вывод, что уровень (`level`) и стаж (`employment_years`)  
оказывают наибольшее влияние на решение модели.

---

💡 Таким образом, добавление инженерных признаков позволило повысить качество прогноза  
и получить модель, пригодную для **HR-аналитики и раннего выявления рисков увольнения**.

### Выводы

После обучения и сравнения нескольких моделей — **Logistic Regression**, **Decision Tree** и **Random Forest** —  
лучший результат показала **модель RandomForestClassifier**, достигнув **ROC-AUC = 0.9310** на тестовой выборке,  
что превышает установленный критерий качества (**≥ 0.91**).

📊 Кросс-валидация показала устойчивость результатов (ROC-AUC = 0.9388),  
а проверка с помощью `DummyClassifier` подтвердила адекватность модели —  
качество значительно выше случайных и константных предсказаний.

🔍 Наибольший вклад в качество предсказаний внесли признаки:
- `job_satisfaction_rate` — уровень удовлетворённости работой,  
- `supervisor_evaluation` — оценка руководителя,  
- `employment_years` — стаж в компании,  
- `salary` — уровень зарплаты,  
- `level` — должностной уровень сотрудника.  

💡 Таким образом, **модель RandomForestClassifier** успешно решает задачу прогнозирования увольнений,  
демонстрируя высокую точность, устойчивость к дисбалансу классов и хорошую обобщающую способность.



## Общий вывод

Передо мной стояло две взаимосвязанные задачи:

1. Построить модель, которая предсказывает **уровень удовлетворённости сотрудника** работой на основе данных компании.  
2. Используя результаты первой модели, создать вторую — для **прогнозирования увольнения сотрудника**.  

Для бизнеса эти задачи критически важны: уровень удовлетворённости напрямую влияет на текучесть кадров,  
а увольнение ценных специалистов влечёт финансовые и организационные риски.  
Заранее выявляя сотрудников с высоким риском ухода, HR-аналитики могут принимать превентивные меры  
и снижать отток персонала.

---

### Этапы работы

- **Загрузка и исследование данных:**  
  Выполнен первичный анализ структуры, проверены пропуски, типы данных и дубликаты.  

- **Предобработка:**  
  Настроены пайплайны с очисткой строк, масштабированием числовых признаков  
  и кодированием категориальных переменных (OrdinalEncoder, OneHotEncoder) и LabelEncoder (для целевого признака).  

- **Исследовательский анализ:**  
  Изучены распределения, корреляции и зависимости между признаками.  
  Подтверждено влияние удовлетворённости (`job_satisfaction_rate`), стажа, уровня и зарплаты на увольнение.  

- **Обучение моделей:**  
  Для первой задачи — регрессионные модели (`Ridge`, `DecisionTreeRegressor`, `RandomForestRegressor`).  
  Для второй — классификаторы (`LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`).  
  Лучшая модель — **RandomForestClassifier**, показала **ROC-AUC = 0.93** на тестовой выборке,  
  что превышает установленный критерий качества (≥ 0.91).

---

### Выводы и рекомендации для бизнеса

- Наибольшая доля увольнений наблюдается в отделах **sales** и **technology** —  
  рекомендуется провести дополнительные опросы удовлетворённости и условий труда.  
- Около **80%** уволившихся — сотрудники уровня *junior* со стажем до трёх лет.  
  Стоит развивать программы адаптации и наставничества.  
- Почти **все уволившиеся** не получали повышения за последний год —  
  необходима прозрачная система карьерного роста.  
- Зарплаты оставшихся сотрудников в среднем **на 50–60% выше**, чем у уволившихся,  
  что указывает на значимость финансовой мотивации.  
- Подтверждена выраженная зависимость: **чем ниже удовлетворённость, тем выше риск увольнения**.  
  Модель позволяет выявлять таких сотрудников заранее и принимать меры (обратная связь, развитие, бонусы).  

---

Таким образом, обе модели образуют **единый HR-аналитический инструмент**:  
первая оценивает **вовлечённость и удовлетворённость**,  
вторая — **прогнозирует риск увольнения**.  
Совместное использование этих решений помогает компании **действовать проактивно**,  
снижать текучесть кадров и формировать устойчивую, мотивированную команду.

